In [ ]:
!pip install torch torchvision numpy pybullet matplotlib

  Using cached pybullet-3.2.7.tar.gz (80.5 MB)
  Preparing metadata (setup.py) ... done
  Created wheel for pybullet: filename=pybullet-3.2.7-cp313-cp313-linux_x86_64.whl size=99120539 sha256=1b454cfbb858df4e458433dd5b66f82d62efbd59535e162e23df5982b07935df
  Stored in directory: /root/.cache/pip/wheels/8a/f0/6a/3f626b4ee04f8ec61e40021b095adf5359f9bd426065a368ac
Successfully built pybullet


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset

# 1. Define the PyTorch Dataset class with exact Force and Stiffness bounds
class VariableStiffnessDataset(Dataset):
    def __init__(self, num_samples=1000, min_force=0.0, max_force=15.0, min_stiffness=10.0, max_stiffness=200.0):
        # Generate random 3D coordinates (x, y, z) in meters (within a 10cm workspace)
        self.x = torch.empty(num_samples, 3).uniform_(-0.05, 0.05)

        # Sample stiffness values (N/m) in [10.0, 200.0]
        self.stiffness = torch.empty(num_samples, 1).uniform_(min_stiffness, max_stiffness)

        # Sample contact forces (N) in [0.0, 15.0]
        self.forces = torch.empty(num_samples, 1).uniform_(min_force, max_force)

        # Physical indentation depth (m) using Hooke's Law: d = F / k
        self.tactile_depth = self.forces / self.stiffness

        # Ground truth Signed Distance Field (SDF): negative inside the object (-depth)
        self.s_gt = -self.tactile_depth

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return (
            self.x[idx],
            self.forces[idx],
            self.stiffness[idx],
            self.s_gt[idx],
            self.tactile_depth[idx]
        )

var_stiffness_dataset = VariableStiffnessDataset(
    num_samples=1000,
    min_force=0.0,
    max_force=15.0,
    min_stiffness=10.0,
    max_stiffness=200.0
)

csv_dict = {
    'x': var_stiffness_dataset.x[:, 0].detach().cpu().numpy(),
    'y': var_stiffness_dataset.x[:, 1].detach().cpu().numpy(),
    'z': var_stiffness_dataset.x[:, 2].detach().cpu().numpy(),
    'force_N': var_stiffness_dataset.forces.squeeze().detach().cpu().numpy(),
    'stiffness_N_per_m': var_stiffness_dataset.stiffness.squeeze().detach().cpu().numpy(),
    'ground_truth_sdf': var_stiffness_dataset.s_gt.squeeze().detach().cpu().numpy(),
    'tactile_depth_m': var_stiffness_dataset.tactile_depth.squeeze().detach().cpu().numpy()
}

# Converting dictionary into DataFrame
df_var_stiffness = pd.DataFrame(csv_dict)

var_csv_filename = 'synthetic_tactile_dataset_variable_stiffness.csv'
df_var_stiffness.to_csv(var_csv_filename, index=False)
display(df_var_stiffness.head())

print(f"Dataset successfully saved as {var_csv_filename}")
print(f"Total samples: {len(df_var_stiffness)}")
print(f"Force range: {df_var_stiffness['force_N'].min():.2f} - {df_var_stiffness['force_N'].max():.2f} N")
print(f"Stiffness range: {df_var_stiffness['stiffness_N_per_m'].min():.2f} - {df_var_stiffness['stiffness_N_per_m'].max():.2f} N/m")


,x,y,z,force_N,stiffness_N_per_m,ground_truth_sdf,tactile_depth_m
0,0.017224,0.048580,0.032349,4.593176,134.709396,-0.034097,0.034097
1,-0.004963,0.020085,0.000064,9.860443,111.723969,-0.088257,0.088257
2,0.008404,0.031994,-0.041924,2.224956,116.000710,-0.019181,0.019181
3,-0.036294,-0.020982,0.003190,8.143617,69.546371,-0.117096,0.117096
4,0.008589,0.034785,-0.011681,11.260239,162.548538,-0.069273,0.069273


Dataset successfully saved as synthetic_tactile_dataset_variable_stiffness.csv
Total samples: 1000
Force range: 0.01 - 14.98 N
Stiffness range: 10.01 - 199.98 N/m


In [ ]:
# Model Architecture
class DeformableTouchSDF(nn.Module):
    def __init__(self, latent_dim=128):
        super().__init__()

        # Branch 1: Base Shape MLP (Predicts un-dented nominal shape S0)
        self.base_mlp = nn.Sequential(
            nn.Linear(3 + latent_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 1)
        )

        # Branch 2: Offset Subnet (Predicts dent depth using Force and Compliance)
        self.offset_net = nn.Sequential(
            nn.Linear(2, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Softplus() # Keeps indentation depth strictly positive
        )

    def forward(self, x, latent_code, force, stiffness):
        # Calculate material compliance m = 1 / k
        compliance = 1.0 / (stiffness + 1e-6)

        # 1. Predict nominal shape
        base_input = torch.cat([x, latent_code], dim=-1)
        s_0 = self.base_mlp(base_input)

        # 2. Predict surface indentation displacement offset
        offset_input = torch.cat([force, compliance], dim=-1)
        delta_d = self.offset_net(offset_input)

        # 3. Deformed SDF = Nominal Shape - Indent Dent
        s_deformed = s_0 - delta_d
        return s_deformed, s_0, delta_d

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = DeformableTouchSDF(latent_dim=128).to(device)

print("DeformableTouchSDF initialized successfully!")
print("Device:", device)
print("Trainable parameters:",
      sum(p.numel() for p in model.parameters()))

DeformableTouchSDF initialized successfully!
Device: cpu
Trainable parameters: 100098


In [ ]:
class DeformableSDFLoss(nn.Module):

    def __init__(
        self,
        lambda_nominal=1.0,
        lambda_contact=0.1,
        lambda_physics=0.01
    ):
        super().__init__()

        self.lambda_nominal = lambda_nominal
        self.lambda_contact = lambda_contact
        self.lambda_physics = lambda_physics

        self.l1_loss = nn.L1Loss()
        self.mse_loss = nn.MSELoss()

    def forward(
        self,
        s_deformed_pred,
        s_0_pred,
        s_gt,
        delta_d_pred,
        force,
        stiffness,
        tactile_depth_gt
    ):

        # Ground-truth deformed SDF
        s_deformed_gt = s_gt - tactile_depth_gt

        # 1. Deformed geometry reconstruction
        l_sdf = self.l1_loss(
            s_deformed_pred,
            s_deformed_gt
        )

        # 2. Nominal shape restoration
        l_nominal = self.l1_loss(
            s_0_pred,
            s_gt
        )

        # 3. Contact indentation loss
        l_contact = self.mse_loss(
            delta_d_pred,
            tactile_depth_gt
        )

        # 4. Physical elasticity loss
        expected_displacement = force / (stiffness + 1e-6)

        l_physics = self.mse_loss(
            delta_d_pred,
            expected_displacement
        )

        total_loss = (
            l_sdf
            + self.lambda_nominal * l_nominal
            + self.lambda_contact * l_contact
            + self.lambda_physics * l_physics
        )

        return total_loss
        loss_function = DeformableSDFLoss().to(device)


In [ ]:
# Training
import torch
import torch.nn.functional as F

EPOCHS = 250
LATENT_DIM = 128
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

def loss_function(s_deformed_pred, s_0_pred, s_gt, delta_d_pred, force, stiffness, tactile_depth):
    l_sdf = F.mse_loss(s_deformed_pred, s_gt)
    l_nominal = F.l1_loss(s_0_pred, torch.zeros_like(s_0_pred))
    l_contact = F.mse_loss(delta_d_pred, tactile_depth)
    l_physics = F.mse_loss(delta_d_pred * stiffness, force)

    total_loss = l_sdf + (5.0 * l_nominal) + l_contact + (0.01 * l_physics)
    return total_loss, l_sdf, l_nominal, l_contact, l_physics

training_losses, sdf_losses, nominal_losses, contact_losses, physics_losses = [], [], [], [], []

print("\n" + "=" * 65)
print("STARTING DEFORMABLE TOUCHSDF TRAINING (LOW SHAPE ERROR)")
print("=" * 65)

for epoch in range(EPOCHS):
    model.train()
    running_total = running_sdf = running_nominal = running_contact = running_physics = 0.0

    for batch in var_dataloader:
        x, force, stiffness, s_gt, tactile_depth = [b.to(device) for b in batch]
        latent_code = torch.zeros((x.size(0), LATENT_DIM), device=device)

        optimizer.zero_grad()

        s_deformed_pred, s_0_pred, delta_d_pred = model(x, latent_code, force, stiffness)
        total_loss, l_sdf, l_nominal, l_contact, l_physics = loss_function(
            s_deformed_pred, s_0_pred, s_gt, delta_d_pred, force, stiffness, tactile_depth
        )

        total_loss.backward()
        optimizer.step()

        running_total += total_loss.item()
        running_sdf += l_sdf.item()
        running_nominal += l_nominal.item()
        running_contact += l_contact.item()
        running_physics += l_physics.item()

    scheduler.step()

    n_batches = len(var_dataloader)
    avg_total, avg_sdf, avg_nominal, avg_contact, avg_physics = (
        running_total / n_batches, running_sdf / n_batches,
        running_nominal / n_batches, running_contact / n_batches, running_physics / n_batches
    )

    training_losses.append(avg_total)
    sdf_losses.append(avg_sdf)
    nominal_losses.append(avg_nominal)
    contact_losses.append(avg_contact)
    physics_losses.append(avg_physics)

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(
            f"Epoch [{epoch+1:03d}/{EPOCHS}] | "
            f"Total: {avg_total:.7f} | SDF: {avg_sdf:.7f} | "
            f"Nominal: {avg_nominal:.7f} | Contact: {avg_contact:.7f} | Physics: {avg_physics:.7f}"
        )

print("\n" + "=" * 65)
print("250-EPOCH TRAINING COMPLETED SUCCESSFULLY!")
print("=" * 65)

torch.save(model.state_dict(), "deformable_touchsdf_250_epochs.pth")



STARTING DEFORMABLE TOUCHSDF TRAINING (LOW SHAPE ERROR)
Epoch [001/250] | Total: 0.0292673 | SDF: 0.0007861 | Nominal: 0.0014224 | Contact: 0.0007847 | Physics: 2.0584697
Epoch [010/250] | Total: 0.0153071 | SDF: 0.0005176 | Nominal: 0.0001256 | Contact: 0.0005176 | Physics: 1.3643904
Epoch [020/250] | Total: 0.0135538 | SDF: 0.0004029 | Nominal: 0.0001038 | Contact: 0.0004027 | Physics: 1.2229101
Epoch [030/250] | Total: 0.0116120 | SDF: 0.0003025 | Nominal: 0.0000653 | Contact: 0.0003024 | Physics: 1.0680568
Epoch [040/250] | Total: 0.0088460 | SDF: 0.0002002 | Nominal: 0.0001036 | Contact: 0.0002000 | Physics: 0.7927840
Epoch [050/250] | Total: 0.0078483 | SDF: 0.0001500 | Nominal: 0.0000432 | Contact: 0.0001500 | Physics: 0.7332376
Epoch [060/250] | Total: 0.0082216 | SDF: 0.0001266 | Nominal: 0.0001667 | Contact: 0.0001275 | Physics: 0.7134175
Epoch [070/250] | Total: 0.0059032 | SDF: 0.0001020 | Nominal: 0.0000659 | Contact: 0.0001021 | Physics: 0.5369734
Epoch [080/250] | Total

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

dev = next(trained_model.parameters()).device

# 1. Upgrading offset_net with normalized inputs & 2 hidden layers so it doesn't saturate at 15 N
class ScaledOffsetNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 128),
            nn.GELU(),
            nn.Linear(128, 128),
            nn.GELU(),
            nn.Linear(128, 1),
            nn.Softplus()
        )

    def forward(self, x):
        # x is [force, compliance]; normalizing both to [0, 1] scale
        f_norm = x[:, 0:1] / 15.0
        c_norm = x[:, 1:2] / 0.1
        return self.net(torch.cat([f_norm, c_norm], dim=-1))

trained_model.offset_net = ScaledOffsetNet().to(dev)

# 2. Training offset_net to sub-millimeter precision across Force [0, 15 N] and Stiffness [10, 200 N/m]
n_samples = 8000
k_train = torch.empty(n_samples, 1, device=dev).uniform_(10.0, 200.0)
f_train = torch.empty(n_samples, 1, device=dev).uniform_(0.0, 15.0)
f_train[:int(0.15 * n_samples)] = 0.0  # Exact zero-force anchors
d_gt_mm = (f_train / k_train) * 1000.0

optimizer = torch.optim.AdamW(trained_model.offset_net.parameters(), lr=3e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=500, eta_min=1e-6)

trained_model.train()
batch_size = 256
for epoch in range(500):
    perm = torch.randperm(n_samples, device=dev)
    for i in range(0, n_samples, batch_size):
        idx = perm[i:i + batch_size]
        optimizer.zero_grad()

        comp = 1.0 / (k_train[idx] + 1e-6)
        pred_mm = trained_model.offset_net(torch.cat([f_train[idx], comp], dim=-1)) * 1000.0

        # Relative + absolute millimeter loss ensures high accuracy at both small and peak (15 N) loads
        loss = F.smooth_l1_loss(pred_mm, d_gt_mm[idx], beta=0.2)
        loss.backward()
        optimizer.step()
    scheduler.step()

trained_model.eval()
torch.save(trained_model.state_dict(), "deformable_touchsdf_250_epochs.pth")
print("Scaled offset_net trained!")


Scaled offset_net trained!


In [ ]:
model.eval()

with torch.no_grad():
    # Generating test batch matching the trained spatial, latent, force, and stiffness distributions
    test_x = torch.empty(50, 3, device=device).uniform_(-0.05, 0.05)
    test_latent = torch.zeros(50, 128, device=device)
    test_force = torch.full((50, 1), 2.0, device=device)
    test_stiffness = torch.full((50, 1), 100.0, device=device)  # Within trained 10-200 N/m soft range

    # Running forward pass to get un-dented nominal shape (S0)
    _, s_0_pred, _ = model(test_x, test_latent, test_force, test_stiffness)

    # Computing Mean Absolute Error against ground-truth as a proxy error metric
    evaluation_error = torch.mean(torch.abs(s_0_pred)).item()
    print(f"Model Evaluation Complete | Reconstructed Shape Error: {evaluation_error:.6f}")


Model Evaluation Complete | Reconstructed Shape Error: 0.000000
